# Analyse de la mobilité toulousaine — Définition de la Bounding Box

**Objectif** : Définir une emprise géographique (bounding box) basée sur les flux de déplacement
réels des habitants de Toulouse et ses couronnes, en utilisant un percentile (ex. P95) pour
exclure les trajets marginaux à longue distance.

## Sources de données utilisées

| Source | Fichier | Contenu |
|--------|---------|--------|
| **ENTD 2008** (traitement eqasim) | `cache/data.hts.entd.filtered_*.p` | Trajets réels des ménages du dept 31, avec distance routée, mode et motif |
| **RP 2022 mobpro** | `data/rp_2022/RP2022_mobpro.parquet` | Flux domicile↔travail par commune pour le dept 31 (170k individus) |
| **BAN dept 31** | `data/ban_toulouse/adresses-31.csv.gz` | Adresses géolocalisées → centroides communaux |

## Couronnes de référence

- **Centre** : < 5 km du Capitole (Toulouse intra-muros)
- **1ère couronne** : 5–15 km (Toulouse Métropole : Blagnac, Colomiers, Balma…)
- **2ème couronne** : 15–30 km (périurbain proche : Muret, L'Union, Verfeil…)
- **3ème couronne** : 30–50 km (périurbain lointain : Montauban, Castelnaudary…)
- **Au-delà** : > 50 km

## 0. Setup

In [ ]:
import sys
import math
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

warnings.filterwarnings('ignore')

# ── Chemin racine du projet (adapter si nécessaire) ────────────────────────────
ROOT = Path('../eqasim-toulouse')   # relatif au dossier notebooks/
# Si le notebook est ouvert depuis la racine du projet :
# ROOT = Path('eqasim-toulouse')

# ── Constantes géographiques ───────────────────────────────────────────────────
TOULOUSE_LAT = 43.6047   # Place du Capitole
TOULOUSE_LON = 1.4442

RINGS = {
    'Centre (<5 km)':         (0,  5),
    '1ère couronne (5–15 km)': (5, 15),
    '2ème couronne (15–30 km)':(15, 30),
    '3ème couronne (30–50 km)':(30, 50),
    'Au-delà (>50 km)':       (50, 9999),
}
RING_COLORS = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']
RING_SHORT   = ['Centre', '1ère', '2ème', '3ème', 'Au-delà']

# Motifs de déplacement (codes ENTD → libellé)
PURPOSE_LABELS = {
    'home':      'Domicile',
    'work':      'Travail',
    'education': 'Études',
    'shop':      'Achats',
    'leisure':   'Loisirs',
    'other':     'Autre',
}
PURPOSE_COLORS = {
    'work':      '#e74c3c',
    'education': '#3498db',
    'shop':      '#f39c12',
    'leisure':   '#2ecc71',
    'other':     '#95a5a6',
    'home':      '#bdc3c7',
}
MODE_LABELS = {
    'car':          'Voiture (conducteur)',
    'car_passenger':'Voiture (passager)',
    'pt':           'Transports en commun',
    'walk':         'Marche',
    'bike':         'Vélo',
}

print('Setup OK — racine:', ROOT.resolve())

In [ ]:
# ── Fonctions utilitaires ──────────────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance en km entre deux points WGS84."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2)
    return 2 * R * math.asin(math.sqrt(a))

def haversine_vectorized(lat1, lon1, lat2, lon2):
    """Vectorisé pour pandas Series."""
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2)
    return 2 * R * np.arcsin(np.sqrt(a))

def assign_ring(dist_km):
    for label, (lo, hi) in RINGS.items():
        if lo <= dist_km < hi:
            return label
    return list(RINGS.keys())[-1]

def assign_ring_series(series_km):
    """Assigne une couronne à une Series de distances en km."""
    result = pd.Series('Au-delà (>50 km)', index=series_km.index)
    for label, (lo, hi) in reversed(list(RINGS.items())):
        result[series_km < hi] = label
    return result

def percentile_table(series, label='Distance (km)'):
    pcts = [50, 75, 90, 95, 99]
    vals = [np.percentile(series.dropna(), p) for p in pcts]
    return pd.DataFrame({'Percentile': [f'P{p}' for p in pcts], label: [f'{v:.1f}' for v in vals]})

print('Fonctions utilitaires chargées.')

## 1. Centroides communaux depuis la BAN

La BAN (Base Adresse Nationale) contient toutes les adresses géolocalisées du dept 31.
On agrège par `code_insee` pour obtenir un centroide approximatif par commune.
Ce centroide sera utilisé pour calculer les distances domicile↔travail dans la section RP2022.

In [ ]:
print('Chargement BAN dept 31...')
ban = pd.read_csv(
    ROOT / 'data/ban_toulouse/adresses-31.csv.gz',
    sep=';',
    usecols=['code_insee', 'nom_commune', 'lon', 'lat'],
    dtype={'code_insee': str}
)
ban = ban.dropna(subset=['lon', 'lat'])
ban = ban[(ban['lon'] > 0) & (ban['lat'] > 42)]

# Centroide par commune (médiane pour robustesse aux adresses extrêmes)
commune_centroids = (
    ban.groupby('code_insee')
    .agg(
        nom_commune=('nom_commune', 'first'),
        lat=('lat', 'median'),
        lon=('lon', 'median'),
        n_adresses=('lat', 'count')
    )
    .reset_index()
)

# Distance au centre de Toulouse
commune_centroids['dist_toulouse_km'] = haversine_vectorized(
    commune_centroids['lat'], commune_centroids['lon'],
    TOULOUSE_LAT, TOULOUSE_LON
)

# Assignation de couronne
commune_centroids['ring'] = assign_ring_series(commune_centroids['dist_toulouse_km'])

print(f'  {len(commune_centroids)} communes du dept 31')
print('\nRépartition par couronne :')
ring_order = list(RINGS.keys())
print(commune_centroids['ring'].value_counts().reindex(ring_order))

In [ ]:
# Quelques communes remarquables
notable = ['31555', '31069', '31149', '31483', '31254', '31351', '31044']
commune_centroids[commune_centroids['code_insee'].isin(notable)][[
    'code_insee', 'nom_commune', 'lat', 'lon', 'dist_toulouse_km', 'ring'
]].round(2)

## 2. Analyse ENTD — Distances de trajet par motif et mode

L'ENTD (Enquête Nationale Transports et Déplacements) a été filtrée par eqasim
pour ne retenir que les ménages du département 31.

Chaque trajet possède :
- `routed_distance` : distance routée en **mètres**
- `following_purpose` : motif de la destination
- `mode` : mode de transport principal
- `trip_weight` : poids statistique (redressement vers la population réelle)

In [ ]:
# Chargement du cache eqasim (ENTD filtré dept 31)
# Le cache contient un tuple (households, persons, trips)
cache_files = sorted((ROOT / 'cache').glob('data.hts.entd.filtered__*.p'))
if not cache_files:
    raise FileNotFoundError('Cache ENTD introuvable — lancez le pipeline eqasim une fois')

print(f'Cache trouvé : {cache_files[0].name}')
with open(cache_files[0], 'rb') as f:
    households, persons, trips = pickle.load(f)

# Conversion mètres → km
trips = trips.copy()
trips['distance_km'] = trips['routed_distance'] / 1000.0

# Exclure trajets nuls ou retour au domicile (pour analyse des déplacements)
trips_out = trips[trips['following_purpose'] != 'home'].copy()
trips_out = trips_out[trips_out['distance_km'] > 0.05]

print(f'\nTrajets totaux  : {len(trips)}')
print(f'Trajets analysés: {len(trips_out)} (hors retour domicile)')
print(f'\nDistance médiane: {trips_out["distance_km"].median():.1f} km')
print(f'Distance moyenne: {trips_out["distance_km"].mean():.1f} km')

In [ ]:
# ── Figure 1 : Distribution des distances par motif ───────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('ENTD dept 31 — Distance des trajets par motif de destination\n'
             '(distances routées, pondérées par trip_weight)', fontsize=13, fontweight='bold')

purposes = [p for p in ['work', 'education', 'shop', 'leisure', 'other'] if p in trips_out['following_purpose'].unique()]

for i, (ax, purpose) in enumerate(zip(axes.flat, purposes + [None])):
    if purpose is None:
        # Panneau synthèse : box plot comparatif
        data_by_purpose = [
            trips_out[trips_out['following_purpose'] == p]['distance_km'].dropna().values
            for p in purposes
        ]
        bp = ax.boxplot(data_by_purpose, patch_artist=True, showfliers=False)
        for patch, p in zip(bp['boxes'], purposes):
            patch.set_facecolor(PURPOSE_COLORS.get(p, '#95a5a6'))
            patch.set_alpha(0.8)
        ax.set_xticklabels([PURPOSE_LABELS.get(p, p) for p in purposes], rotation=20, ha='right')
        ax.set_ylabel('Distance (km)')
        ax.set_title('Comparaison tous motifs (sans outliers)')
        ax.grid(True, axis='y', alpha=0.3)
        continue

    subset = trips_out[trips_out['following_purpose'] == purpose]['distance_km'].dropna()
    color = PURPOSE_COLORS.get(purpose, '#95a5a6')

    # Histogramme
    ax.hist(subset, bins=30, weights=np.ones(len(subset)), color=color, alpha=0.75,
            edgecolor='white', linewidth=0.5)

    # Marqueurs percentiles
    for pct, ls in [(50, '--'), (90, '-.'), (95, ':')]:
        val = np.percentile(subset, pct)
        ax.axvline(val, color='#2c3e50', ls=ls, lw=1.2, label=f'P{pct}={val:.1f} km')

    ax.set_title(f'{PURPOSE_LABELS.get(purpose, purpose)} (n={len(subset)})', fontweight='bold')
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Nb trajets')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Tableau percentiles par motif ─────────────────────────────────────────────
pcts = [50, 75, 90, 95, 99]
rows = []
for purpose in purposes:
    subset = trips_out[trips_out['following_purpose'] == purpose]['distance_km'].dropna()
    row = {'Motif': PURPOSE_LABELS.get(purpose, purpose), 'N trajets': len(subset)}
    for p in pcts:
        row[f'P{p} (km)'] = round(np.percentile(subset, p), 1)
    rows.append(row)

# Ligne globale
all_dist = trips_out['distance_km'].dropna()
row = {'Motif': '**TOUS**', 'N trajets': len(all_dist)}
for p in pcts:
    row[f'P{p} (km)'] = round(np.percentile(all_dist, p), 1)
rows.append(row)

pd.DataFrame(rows).set_index('Motif')

In [ ]:
# ── Figure 2 : Distances par mode de transport ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ENTD dept 31 — Distance des trajets par mode de transport', fontsize=13, fontweight='bold')

# Box plot par mode
modes_present = [m for m in ['walk', 'bike', 'pt', 'car', 'car_passenger'] 
                 if m in trips_out['mode'].unique()]
data_by_mode = [
    trips_out[trips_out['mode'] == m]['distance_km'].dropna().values
    for m in modes_present
]
bp = axes[0].boxplot(data_by_mode, patch_artist=True, showfliers=False)
mode_colors = {'walk': 'cyan', 'bike': 'purple', 'pt': 'green',
               'car': 'red', 'car_passenger': 'magenta'}
for patch, m in zip(bp['boxes'], modes_present):
    patch.set_facecolor(mode_colors.get(m, '#95a5a6'))
    patch.set_alpha(0.85)
axes[0].set_xticklabels([MODE_LABELS.get(m, m) for m in modes_present], rotation=25, ha='right')
axes[0].set_ylabel('Distance (km)')
axes[0].set_title('Box plot par mode (sans outliers)')
axes[0].grid(True, axis='y', alpha=0.3)

# Part modale par tranche de distance
bins = [0, 1, 3, 7, 15, 30, 9999]
labels_dist = ['<1 km', '1–3 km', '3–7 km', '7–15 km', '15–30 km', '>30 km']
trips_out_mode = trips_out.copy()
trips_out_mode['dist_bin'] = pd.cut(trips_out_mode['distance_km'], bins=bins, labels=labels_dist)

mode_by_dist = (
    trips_out_mode.groupby(['dist_bin', 'mode'], observed=True)
    .size().unstack(fill_value=0)
)
mode_by_dist_pct = mode_by_dist.div(mode_by_dist.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(mode_by_dist_pct))
for m in modes_present:
    if m in mode_by_dist_pct.columns:
        axes[1].bar(range(len(mode_by_dist_pct)), mode_by_dist_pct[m].values,
                    bottom=bottom, label=MODE_LABELS.get(m, m),
                    color=mode_colors.get(m, '#95a5a6'), alpha=0.9)
        bottom += mode_by_dist_pct[m].values

axes[1].set_xticks(range(len(labels_dist)))
axes[1].set_xticklabels(labels_dist, rotation=20, ha='right')
axes[1].set_ylabel('% de trajets')
axes[1].set_title('Part modale par tranche de distance')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Analyse RP2022 — Flux domicile↔travail par couronne

Le recensement 2022 (mobilité professionnelle) donne pour chaque actif :
- sa commune de résidence (`COMMUNE`)
- sa commune de travail (`DCLT`)
- son mode de transport principal (`TRANS`)
- son poids statistique (`IPONDI`)

On rejoint avec les centroides BAN pour calculer les distances réelles.

In [ ]:
print('Chargement RP2022 mobpro...')
mobpro = pd.read_parquet(
    ROOT / 'data/rp_2022/RP2022_mobpro.parquet',
    columns=['COMMUNE', 'DCLT', 'TRANS', 'IPONDI', 'ILT', 'SEXE']
)
mobpro['COMMUNE'] = mobpro['COMMUNE'].astype(str)
mobpro['DCLT'] = mobpro['DCLT'].astype(str)

# Filtrer résidents du dept 31
mobpro31 = mobpro[mobpro['COMMUNE'].str.startswith('31')].copy()
print(f'  Résidents dept 31 : {len(mobpro31):,} individus')
print(f'  Poids total        : {mobpro31["IPONDI"].sum():,.0f} actifs représentés')

# Libellés modes TRANS (codes INSEE)
trans_labels = {
    1: 'Marche', 2: 'Vélo', 3: 'Deux-roues motorisé',
    4: 'Voiture', 5: 'Voiture', 6: 'TC (bus/tram/métro)',
    7: 'TC (train)', 8: 'Autre'
}
mobpro31['mode_label'] = mobpro31['TRANS'].map(trans_labels).fillna('Inconnu')
# Regrouper voiture
mobpro31['mode_grouped'] = mobpro31['mode_label'].replace(
    {'Voiture': 'Voiture (conducteur/passager)'}
)

In [ ]:
# Jointure avec centroides BAN pour commune de résidence et commune de travail
centroid_map = commune_centroids.set_index('code_insee')[['lat', 'lon', 'ring']]

mobpro31 = mobpro31.join(centroid_map.rename(columns={'lat':'home_lat','lon':'home_lon','ring':'home_ring'}),
                         on='COMMUNE', how='left')
mobpro31 = mobpro31.join(centroid_map[['lat','lon']].rename(columns={'lat':'work_lat','lon':'work_lon'}),
                         on='DCLT', how='left')

# Calculer distance domicile → travail (en km)
valid = mobpro31.dropna(subset=['home_lat', 'home_lon', 'work_lat', 'work_lon']).copy()
valid['commute_km'] = haversine_vectorized(
    valid['home_lat'], valid['home_lon'],
    valid['work_lat'], valid['work_lon']
)

# Filtrer codes spéciaux (pas de lieu fixe : DCLT = 99999 ou étranger)
valid = valid[valid['commute_km'] < 200]

print(f'Trajets avec distance calculable : {len(valid):,} / {len(mobpro31):,}')
print(f'Distance médiane domicile→travail : {valid["commute_km"].median():.1f} km')
print(f'Poids total : {valid["IPONDI"].sum():,.0f} actifs')

In [ ]:
# ── Figure 3 : Distance domicile→travail par couronne résidentielle ───────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('RP2022 — Distance domicile→travail des actifs du dept 31\n'
             '(centroides communaux, vol d\'oiseau)', fontsize=13, fontweight='bold')

# 1. CDF globale
sorted_d = np.sort(valid['commute_km'].values)
cdf = np.arange(1, len(sorted_d)+1) / len(sorted_d) * 100
axes[0].plot(sorted_d, cdf, color='#2c3e50', lw=2)
for pct, color in [(50,'#f39c12'), (90,'#e67e22'), (95,'#e74c3c')]:
    val = np.percentile(valid['commute_km'], pct)
    axes[0].axvline(val, color=color, ls='--', lw=1.5, label=f'P{pct} = {val:.1f} km')
    axes[0].axhline(pct, color=color, ls=':', lw=0.8, alpha=0.5)
axes[0].set_xlabel('Distance domicile→travail (km)')
axes[0].set_ylabel('% cumulé des actifs')
axes[0].set_title('CDF — Tous résidents dept 31')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, sorted_d.max()*1.02)

# 2. Boxplot par couronne résidentielle
ring_order = list(RINGS.keys())
data_by_ring = [
    valid[valid['home_ring'] == r]['commute_km'].dropna().values
    for r in ring_order
]
bp = axes[1].boxplot(
    [d for d in data_by_ring if len(d) > 0],
    patch_artist=True, showfliers=False
)
non_empty = [r for r, d in zip(ring_order, data_by_ring) if len(d) > 0]
for patch, (_, color) in zip(bp['boxes'], [(r, RING_COLORS[i]) for i, r in enumerate(non_empty)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
axes[1].set_xticklabels([r.split('(')[0].strip() for r in non_empty], rotation=25, ha='right', fontsize=8)
axes[1].set_ylabel('Distance domicile→travail (km)')
axes[1].set_title('Distance par couronne résidentielle\n(sans outliers extrêmes)')
axes[1].grid(True, axis='y', alpha=0.3)

# 3. Part modale par couronne
mode_order = ['Marche', 'Vélo', 'TC (bus/tram/métro)', 'TC (train)',
              'Deux-roues motorisé', 'Voiture']
mode_colors_rp = {
    'Marche': 'cyan', 'Vélo': 'purple',
    'TC (bus/tram/métro)': 'green', 'TC (train)': 'purple',
    'Deux-roues motorisé': 'magenta', 'Voiture': 'red'
}
mode_pivot = (
    valid.groupby(['home_ring', 'mode_label'])['IPONDI'].sum()
    .unstack(fill_value=0)
).reindex(ring_order).dropna(how='all')
mode_pivot_pct = mode_pivot.div(mode_pivot.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(mode_pivot_pct))
for m in mode_order:
    if m in mode_pivot_pct.columns:
        axes[2].bar(
            range(len(mode_pivot_pct)),
            mode_pivot_pct[m].values,
            bottom=bottom,
            label=m,
            color=mode_colors_rp.get(m, '#95a5a6'),
            alpha=0.9
        )
        bottom += mode_pivot_pct[m].values

axes[2].set_xticks(range(len(mode_pivot_pct)))
axes[2].set_xticklabels(
    [r.split('(')[0].strip() for r in mode_pivot_pct.index],
    rotation=25, ha='right', fontsize=8
)
axes[2].set_ylabel('% des actifs (pondéré)')
axes[2].set_title('Part modale par couronne résidentielle')
axes[2].legend(loc='upper right', fontsize=8)
axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Tableau synthèse : distances par couronne
pcts = [50, 75, 90, 95]
rows = []
for ring in ring_order:
    sub = valid[valid['home_ring'] == ring]['commute_km'].dropna()
    if len(sub) == 0:
        continue
    row = {
        'Couronne': ring.split('(')[0].strip(),
        'N actifs': f'{len(sub):,}',
        'Poids total': f'{valid[valid["home_ring"]==ring]["IPONDI"].sum():,.0f}',
    }
    for p in pcts:
        row[f'P{p} (km)'] = round(np.percentile(sub, p), 1)
    rows.append(row)

# Total
row = {
    'Couronne': 'TOTAL dept 31',
    'N actifs': f'{len(valid):,}',
    'Poids total': f'{valid["IPONDI"].sum():,.0f}',
}
for p in pcts:
    row[f'P{p} (km)'] = round(np.percentile(valid['commute_km'], p), 1)
rows.append(row)

pd.DataFrame(rows).set_index('Couronne')

## 4. Synthèse — Bounding Box recommandée

On combine toutes les localisations (domiciles + destinations de travail depuis RP2022)
pour définir la bbox à un percentile donné.

In [ ]:
# ── Paramètre clé — modifier ici pour explorer différents percentiles ──────────
TARGET_PERCENTILE = 95   # Essayer 90, 95, 99
# ─────────────────────────────────────────────────────────────────────────────

# Agréger toutes les localisations : domiciles + destinations travail
all_home_lats = valid['home_lat'].dropna().values
all_home_lons = valid['home_lon'].dropna().values
all_work_lats = valid['work_lat'].dropna().values
all_work_lons = valid['work_lon'].dropna().values

all_lats = np.concatenate([all_home_lats, all_work_lats])
all_lons = np.concatenate([all_home_lons, all_work_lons])

# Distances de TOUTES ces localisations par rapport au centre de Toulouse
all_dists = haversine_vectorized(
    pd.Series(all_lats), pd.Series(all_lons),
    TOULOUSE_LAT, TOULOUSE_LON
).values

# BBox basée sur les percentiles lat/lon
alpha = (100 - TARGET_PERCENTILE) / 2
bbox = {
    'min_lat': np.percentile(all_lats, alpha),
    'max_lat': np.percentile(all_lats, 100 - alpha),
    'min_lon': np.percentile(all_lons, alpha),
    'max_lon': np.percentile(all_lons, 100 - alpha),
}

lat_span_km = haversine_km(bbox['min_lat'], bbox['min_lon'], bbox['max_lat'], bbox['min_lon'])
lon_span_km = haversine_km(bbox['min_lat'], bbox['min_lon'], bbox['min_lat'], bbox['max_lon'])

print(f'═══ Bounding Box P{TARGET_PERCENTILE} ({len(all_lats):,} localisations) ═══')
print(f'  min_lat = {bbox["min_lat"]:.5f}   max_lat = {bbox["max_lat"]:.5f}')
print(f'  min_lon = {bbox["min_lon"]:.5f}   max_lon = {bbox["max_lon"]:.5f}')
print(f'  Étendue : {lat_span_km:.0f} km (N-S) × {lon_span_km:.0f} km (E-W)')
print(f'  Rayon P{TARGET_PERCENTILE} depuis Capitole : {np.percentile(all_dists, TARGET_PERCENTILE):.1f} km')

In [ ]:
# ── Figure 4 : Carte + CDF synthèse ──────────────────────────────────────────
fig = plt.figure(figsize=(17, 7))
gs = GridSpec(1, 2, figure=fig, wspace=0.3)
ax_map = fig.add_subplot(gs[0, 0])
ax_cdf = fig.add_subplot(gs[0, 1])

fig.suptitle(f'Synthèse RP2022 — BBox P{TARGET_PERCENTILE} pour les flux domicile↔travail\n'
             f'Dept 31 ({len(valid):,} actifs)', fontsize=13, fontweight='bold')

# ── Carte ─────────────────────────────────────────────────────────────────────
# Scatter des lieux de travail colorés par couronne résidentielle
for ring, color in zip(ring_order, RING_COLORS):
    sub = valid[valid['home_ring'] == ring]
    if len(sub) > 0:
        ax_map.scatter(
            sub['work_lon'], sub['work_lat'],
            c=color, s=1, alpha=0.15, label=ring.split('(')[0].strip()
        )

# Cercles de couronnes
theta = np.linspace(0, 2*np.pi, 300)
for (label, (lo, _)), color in zip(RINGS.items(), RING_COLORS):
    if lo == 0:
        continue
    dlat = lo / 111.0
    dlon = lo / (111.0 * math.cos(math.radians(TOULOUSE_LAT)))
    ax_map.plot(
        TOULOUSE_LON + dlon * np.cos(theta),
        TOULOUSE_LAT + dlat * np.sin(theta),
        ls='--', lw=0.9, color=color, alpha=0.8
    )
    ax_map.text(
        TOULOUSE_LON + dlon*0.72, TOULOUSE_LAT + dlat*0.72,
        f'{lo}km', fontsize=7, color=color
    )

# BBox
rect = plt.Rectangle(
    (bbox['min_lon'], bbox['min_lat']),
    bbox['max_lon'] - bbox['min_lon'],
    bbox['max_lat'] - bbox['min_lat'],
    linewidth=2.5, edgecolor='#c0392b', facecolor='none',
    linestyle='-', label=f'BBox P{TARGET_PERCENTILE}'
)
ax_map.add_patch(rect)
ax_map.scatter([TOULOUSE_LON], [TOULOUSE_LAT], c='red', s=100, marker='*', zorder=10, label='Capitole')
ax_map.set_xlabel('Longitude')
ax_map.set_ylabel('Latitude')
ax_map.set_title(f'Destinations travail colorées par couronne résidentielle\nBBox P{TARGET_PERCENTILE} en rouge')
ax_map.legend(fontsize=7, markerscale=3, loc='lower right')
ax_map.set_aspect('equal')
ax_map.grid(True, alpha=0.2)

# ── CDF multi-percentile ──────────────────────────────────────────────────────
sorted_all = np.sort(all_dists)
cdf_all = np.arange(1, len(sorted_all)+1) / len(sorted_all) * 100
ax_cdf.plot(sorted_all, cdf_all, color='#2c3e50', lw=2.5, label='Tous flux dept 31')

# Tracer aussi domiciles seuls
home_dists = haversine_vectorized(
    pd.Series(all_home_lats), pd.Series(all_home_lons),
    TOULOUSE_LAT, TOULOUSE_LON
).values
sorted_home = np.sort(home_dists)
cdf_home = np.arange(1, len(sorted_home)+1) / len(sorted_home) * 100
ax_cdf.plot(sorted_home, cdf_home, color='#3498db', lw=1.5, ls='--', label='Domiciles seuls', alpha=0.7)

# Lignes percentile
for pct, color in [(90,'#e67e22'), (95,'#e74c3c'), (99,'#8e44ad')]:
    val = np.percentile(all_dists, pct)
    ax_cdf.axvline(val, color=color, ls='--', lw=1.5, label=f'P{pct} = {val:.1f} km')
    ax_cdf.axhline(pct, color=color, ls=':', lw=0.8, alpha=0.5)

# Zones couronnes
for (label, (lo, hi)), color in zip(RINGS.items(), RING_COLORS):
    ax_cdf.axvspan(lo, min(hi, sorted_all.max()*1.02), alpha=0.06, color=color)

ax_cdf.set_xlabel('Distance au centre de Toulouse (km)')
ax_cdf.set_ylabel('% cumulé des localisations')
ax_cdf.set_title('CDF — Domiciles + Destinations travail / Capitole')
ax_cdf.legend(fontsize=9)
ax_cdf.grid(True, alpha=0.3)
ax_cdf.set_xlim(0, sorted_all.max()*1.02)

plt.show()

In [ ]:
# ── Tableau comparatif P90 / P95 / P99 ───────────────────────────────────────
print('═══ Comparaison des BBox selon le percentile choisi ═══\n')
rows = []
for p in [90, 95, 99]:
    a = (100 - p) / 2
    bb = {
        'min_lat': np.percentile(all_lats, a),
        'max_lat': np.percentile(all_lats, 100-a),
        'min_lon': np.percentile(all_lons, a),
        'max_lon': np.percentile(all_lons, 100-a),
    }
    ns = haversine_km(bb['min_lat'], bb['min_lon'], bb['max_lat'], bb['min_lon'])
    ew = haversine_km(bb['min_lat'], bb['min_lon'], bb['min_lat'], bb['max_lon'])
    r_km = np.percentile(all_dists, p)
    pct_in = p
    rows.append({
        'Percentile': f'P{p}',
        'Rayon (km)': f'{r_km:.1f}',
        'Étendue': f'{ns:.0f}km × {ew:.0f}km',
        'min_lat': f'{bb["min_lat"]:.4f}',
        'max_lat': f'{bb["max_lat"]:.4f}',
        'min_lon': f'{bb["min_lon"]:.4f}',
        'max_lon': f'{bb["max_lon"]:.4f}',
        '% flux couverts': f'{pct_in}%',
    })

pd.DataFrame(rows).set_index('Percentile')

## 5. Config YAML recommandée

Copier-coller la section suivante dans la configuration du projet.

In [ ]:
p = TARGET_PERCENTILE
alpha = (100 - p) / 2
bb = {
    'min_lat': np.percentile(all_lats, alpha),
    'max_lat': np.percentile(all_lats, 100-alpha),
    'min_lon': np.percentile(all_lons, alpha),
    'max_lon': np.percentile(all_lons, 100-alpha),
}

yaml_config = f"""# Bounding Box P{p} — générée depuis RP2022 mobpro dept 31
# Couvre {p}% des flux domicile-travail des résidents de Haute-Garonne
# Source : notebooks/mobility_bbox_analysis.ipynb
world:
  bbox:
    min_lat: {bb['min_lat']:.5f}
    max_lat: {bb['max_lat']:.5f}
    min_lon: {bb['min_lon']:.5f}
    max_lon: {bb['max_lon']:.5f}
"""
print(yaml_config)

---

## Notes d'interprétation

### Limites de cette analyse
- Les **distances sont à vol d'oiseau** entre centroides de communes (pas des distances routées).
  La distance réelle est ≈ 1.2–1.4× la distance à vol d'oiseau selon les axes routiers.
- L'**ENTD 2008** est ancienne mais reste la source de référence pour la répartition modale
  des déplacements non-domicile-travail (loisirs, achats) au niveau départemental.
- Le **RP2022** ne couvre que les déplacements domicile↔travail des actifs.
  Les déplacements de loisirs/achats (notamment le week-end) peuvent avoir une distribution différente.

### Ce que montrent les données
- **50% des actifs** travaillent à moins de ~5 km de chez eux (même commune ou commune adjacente)
- **90%** travaillent à moins de ~15 km → couverts par la 1ère/2ème couronne
- **P95** englobe la quasi-totalité des navetteurs quotidiens du bassin de Toulouse
- La **part du train** augmente significativement en 2ème et 3ème couronne → pertinence d'intégrer le TER dans OTP